# 01 -- Data Preparation, Cleaning & ComBat Harmonization

**Corresponds to:** Manuscript Sec.3.1 (Participants), Sec.3.2 (Integrated Diffusion Framework), Sec.3.4.4 (Confound Control)

This notebook loads the raw feature matrix, imputes missing values, maps diagnosis codes to binary labels, and applies ComBat harmonization to remove site-specific scanner effects while preserving biological variance.

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import os, warnings, sys
warnings.filterwarnings('ignore')
np.random.seed(41)

# Paths
DATA_DIR = os.path.join("data")
DATA_FILE = os.path.join(DATA_DIR, "finaldata.csv")
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Try ComBat
try:
    from neuroCombat import neuroCombat
    COMBAT_AVAILABLE = True
    print("neuroCombat: available")
except ImportError:
    COMBAT_AVAILABLE = False
    print("neuroCombat: NOT available -- install with: pip install neuroCombat")

## 1. Load Raw Data

In [ ]:
df = pd.read_csv(DATA_FILE)
print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns")

# Identify feature columns (all diffusion metrics start at column index 58)
input_features = df.columns.tolist()[58:]
target_col = "Diagnosis_three_categories_code"
site_col = "scan_site_text"
print(f"Feature columns: {len(input_features)}")
print(f"First feature: {input_features[0]}")
print(f"Last feature:  {input_features[-1]}")

## 2. Diagnosis Mapping

Two classification tasks:
- **Patient vs Control:** Diagnoses 1 (schizophrenia), 2 (schizoaffective), 3 (non-schizophrenia psychosis) -> 'Patient'; 0 -> 'Control'
- **SCZ vs Non-SCZ:** Diagnoses 1, 2 -> 'SCZ'; 3 -> 'Non SCZ'; 0 (controls) excluded

In [ ]:
def map_patient_vs_control(val):
    if val in [1, 2, 3]: return "Patient"
    elif val == 0: return "Control"
    return "Remove"

def map_scz_vs_non_scz(val):
    if val in [1, 2]: return "SCZ"
    elif val == 3: return "Non SCZ"
    return "Remove"

# Drop rows missing target or site
df_clean = df.dropna(subset=[target_col, site_col]).copy()
print(f"After dropping NA target/site: {df_clean.shape[0]} rows")

# Compute both mappings
df_clean['diag_pvc'] = df_clean[target_col].apply(map_patient_vs_control)
df_clean['diag_scz'] = df_clean[target_col].apply(map_scz_vs_non_scz)

for label_col, task in [('diag_pvc', 'Patient vs Control'), ('diag_scz', 'SCZ vs Non-SCZ')]:
    subset = df_clean[df_clean[label_col] != 'Remove']
    print(f"\n{task}:")
    print(f"  Total: {len(subset)}")
    print(f"  Class distribution:\n{subset[label_col].value_counts()}")

## 3. Handle Missing Feature Values

We use median imputation to handle any missing diffusion metrics. This is applied *before* ComBat and cross-validation since imputation is an unsupervised preprocessing step that does not leak information.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_raw = df_clean[input_features].values
X_imputed = imputer.fit_transform(X_raw)

# Create cleaned dataframe
df_clean[input_features] = X_imputed
print(f"Missing values after imputation: {np.sum(np.isnan(X_imputed))}")

# Per-feature stats
feature_stats = pd.DataFrame({
    'feature': input_features,
    'mean': np.nanmean(X_raw, axis=0),
    'std': np.nanstd(X_raw, axis=0),
    'missing_pct': np.isnan(X_raw).mean(axis=0) * 100
})
print(f"\nFeatures with >5% missing:")
print(feature_stats[feature_stats['missing_pct'] > 5].to_string(index=False))

## 4. ComBat Harmonization

**Manuscript Sec.3.4.4:** ComBat harmonization removes additive and multiplicative site-specific scanner effects while preserving diagnostic variance. This is critical because our data comes from multiple acquisition sites (WashU, Indiana University, McLean, Brigham & Women's Hospital).

The harmonization is applied *before* cross-validation since it is an unsupervised batch-effect correction that does not use label information (the disease status is provided only to preserve biological variance).

In [ ]:
sites = df_clean[site_col].astype(str).values
unique_sites = np.unique(sites)
print(f"Sites in data: {unique_sites}")
print(f"Site counts:\n{df_clean[site_col].value_counts()}")

X_harmonized = X_imputed.copy()
if COMBAT_AVAILABLE and len(unique_sites) > 1:
    for task_name in ['pvc', 'scz']:
        if task_name == 'pvc':
            y = (df_clean['diag_pvc'] == 'Patient').astype(int).values
        else:
            y = (df_clean['diag_scz'] == 'SCZ').astype(int).values

        covars = pd.DataFrame({'batch': sites, 'disease': y})
        combat_out = neuroCombat(
            dat=X_imputed.T, covars=covars,
            batch_col='batch', categorical_cols=['disease']
        )

    X_harmonized = combat_out['data'].T
    df_clean[[f"harm_{f}" for f in input_features]] = X_harmonized
    print(f"\nComBat harmonization applied successfully.")
    print(f"Output shape: {X_harmonized.shape}")

    # Show effect on a sample feature
    sample_feat = input_features[0]
    print(f"\nEffect on {sample_feat}:")
    print(f"  Before -- mean: {X_imputed[:,0].mean():.4f}, std: {X_imputed[:,0].std():.4f}")
    print(f"  After  -- mean: {X_harmonized[:,0].mean():.4f}, std: {X_harmonized[:,0].std():.4f}")
else:
    print("\nComBat not available or single site -- skipping harmonization.")
    for f in input_features:
        df_clean[f"harm_{f}"] = df_clean[f]

## 5. Save Processed Data

We save separate copies for each classification task and a harmonized + unharmonized version for the robustness analysis.

In [ ]:
# For Patient vs Control
pvc_mask = df_clean['diag_pvc'] != 'Remove'
df_pvc = df_clean[pvc_mask].copy()
df_pvc.to_pickle(os.path.join(OUTPUT_DIR, "data_pvc.pkl"))
print(f"Patient vs Control: {len(df_pvc)} samples saved")

# For SCZ vs Non-SCZ
scz_mask = df_clean['diag_scz'] != 'Remove'
df_scz = df_clean[scz_mask].copy()
df_scz.to_pickle(os.path.join(OUTPUT_DIR, "data_scz.pkl"))
print(f"SCZ vs Non-SCZ: {len(df_scz)} samples saved")

# Full data with harmonized features
df_clean.to_pickle(os.path.join(OUTPUT_DIR, "data_full.pkl"))
print(f"Full data ({len(df_clean)} samples) saved")

# Save feature list
with open(os.path.join(OUTPUT_DIR, "feature_names.txt"), "w") as f:
    f.write("\n".join(input_features))
print("\n✓ Data preparation complete.")